In [ ]:
import scanpy as sc
from lets_plot import *

import cellestial as cl

In [ ]:
data = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
vl =cl.violin(data,key="CD3D",fill="cell_type_lvl1",threshold=0.1)

In [ ]:
cl.retrieve(vl)

In [ ]:
results

In [ ]:
from itertools import combinations

import polars as pl
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

df = cl.retrieve(plot=vl)
groups = df["cell_type_lvl1"].unique().to_list()

rows = []
for a, b in combinations(groups, 2):
    values_a = df.filter(pl.col("cell_type_lvl1") == a)["CD3D"].to_numpy()
    values_b = df.filter(pl.col("cell_type_lvl1") == b)["CD3D"].to_numpy()
    _, pvalue = mannwhitneyu(values_a, values_b, alternative="two-sided")
    rows.append({"xmin": a, "xmax": b, "pvalue": pvalue})

brackets = pl.DataFrame(rows)

_, pvalue_adj, *_ = multipletests(brackets["pvalue"].to_numpy(), method="fdr_bh")

def stars(p: float) -> str:
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

y_top = df["CD3D"].max()
y_step = (y_top - df["CD3D"].min()) * 0.08

brackets = brackets.with_columns(
    pvalue_adj=pl.Series(pvalue_adj),
).with_columns(
    label=pl.Series([stars(p) for p in pvalue_adj]),
    y=pl.Series([y_top + y_step * (i + 1) for i in range(len(brackets))]),
)


In [ ]:
brackets

In [ ]:
vl + geom_bracket(
    aes(xmin="xmin", xmax="xmax", y="y", label="label"),
    data=brackets,
)
